In [0]:
# ============================================================
# nb_05_workflow_setup_guide
# Reference notebook — not executed as part of any workflow.
# Documents workflow setup, enable patterns, re-run patterns,
# and load_mode reference.
# ============================================================

## Workflow Setup Guide
### One Databricks Workflow per process_group
Each workflow has three tasks in sequence:

Task 1: init     → nb_02_workflow_init
Task 2: dispatch → nb_03_dispatcher     (depends on: init)
Task 3: summary  → nb_04_workflow_summary (depends on: dispatch)
```
Set parameters at the JOB level (not per task):
```
process_group  = TPCH_foreign_catalog
admin_catalog  = sandbox
config_schema  = migration_config
max_workers    = 8
```

In [0]:
dbutils.widgets.text("admin_catalog",  "sandbox")
dbutils.widgets.text("config_schema",  "migration_config")
dbutils.widgets.text("process_group",  "TPCH_foreign_catalog")

ADMIN_CATALOG = dbutils.widgets.get("admin_catalog")
CONFIG_SCHEMA = dbutils.widgets.get("config_schema")
PROCESS_GROUP = dbutils.widgets.get("process_group")
CONFIG_TABLE  = f"{ADMIN_CATALOG}.{CONFIG_SCHEMA}.table_migration_config"
CONN_TABLE    = f"{ADMIN_CATALOG}.{CONFIG_SCHEMA}.source_connection_config"

In [0]:
# Current config state for selected process group
display(spark.sql(f"""
    SELECT
        table_id, src_database, src_schema, src_table,
        target_schema || '.' || target_table AS target,
        load_mode, incremental_col, primary_keys,
        process_group, priority, enabled,
        last_run_status, last_run_at,
        last_sf_row_count, last_delta_count,
        last_loaded_value
    FROM {CONFIG_TABLE}
    WHERE process_group = '{PROCESS_GROUP}'
    ORDER BY priority, table_id
"""))

In [0]:
# Connection registry
display(spark.sql(f"""
    SELECT * FROM {CONN_TABLE}
    ORDER BY source_type, connection_name
"""))

## Enable tables by process group
```sql
UPDATE sandbox.migration_config.table_migration_config
SET enabled = true
WHERE process_group = 'TPCH_foreign_catalog';
```

## Re-run failed tables
```sql
UPDATE sandbox.migration_config.table_migration_config
SET last_run_status = 'PENDING'
WHERE process_group   = 'TPCH_foreign_catalog'
  AND last_run_status = 'FAILED';
```

## Full reset (clears watermark — reloads all data)
```sql
UPDATE sandbox.migration_config.table_migration_config
SET last_run_status    = 'PENDING',
    last_loaded_value   = NULL,
    last_sf_row_count   = NULL,
    last_delta_count    = NULL,
    notes               = NULL
WHERE process_group = 'TPCH_foreign_catalog';
```

## Process groups and workflows
| process_group | Workflow | max_workers | Notes |
|---|---|---|---|
| TPCH_foreign_catalog | TPCH_foreign_catalog_wf | 8 | All samples.tpch tables via foreign catalog |
| TPCH_autoloader | TPCH_autoloader_wf | 8 | CSV files in UC Volume via Autoloader |
| VOLUME_copy_into | VOLUME_copy_into_wf | 1 | Parquet files in UC Volume via COPY INTO |
| PG_NEON_jdbc | PG_NEON_wf | 8 | Neon PostgreSQL via foreign catalog |
## load_mode reference
| load_mode | load_strategy | Source read | Best for |
|---|---|---|---|
| overwrite | full | Foreign catalog or JDBC | Reference tables, small-medium tables |
| append | full | Foreign catalog or JDBC | Insert-only sources, audit logs |
| merge | incremental | Foreign catalog + watermark | Fact/dim tables with updates |
| copy_into | full | UC Volume (volume) or ADLS blob (snowflake) | Large tables, bulk loads |
| autoloader | incremental | UC Volume CSV/JSON/Parquet | Continuous or scheduled file ingestion |

 ## Adding a new source system

 1. Insert a row into source_connection_config:
 ```sql
 INSERT INTO sandbox.migration_config.source_connection_config VALUES (
     'my_new_source',      -- connection_name (matches src_database)
     'sqlserver',          -- source_type
     'foreign_catalog',    -- connection_method
     'my_new_source',      -- catalog_name (UC registered catalog)
     NULL,                 -- jdbc_url_template
     NULL,                 -- driver_class
     NULL,                 -- secret_scope
     NULL,                 -- volume_base_path
     false,                -- ssl_enabled
     NULL,                 -- extra_options
     true,                 -- enabled
     'New SQL Server source via foreign catalog'
 );
 ```

 2. Insert rows into table_migration_config with src_database = 'my_new_source'
 3. No code changes required in any notebook